# INFO-H-515 Project: Task 1 - Preprocessing, Tokenization, and Embeddings
**Team:** 7
**Objective:** Ingest course PDFs, clean text, perform chunking with metadata, and generate embeddings using SBERT.

In [ ]:
# Import necessary libraries
import io
import re
from pyspark.sql import SparkSession
import pypdf


In [31]:
# Task 1 Parameters (as required by project specifications)
parallelism = 4
chunk_size = 250
chunk_overlap = 50
embedding_strategy = "SBERT"
input_dir = "data/data_raw/*.pdf"
output_dir = "data/data_processed/embedded_chunks.parquet"

In [21]:
# 1. Function to extract and clean PDF pages while keeping page numbers
def extract_pages_from_pdf(file_path_and_content):
    """
    Extracts text page by page to preserve page number metadata.
    Returns a list of tuples: (file_path, page_num, clean_text)
    """
    file_path, file_content = file_path_and_content
    pages_data = []
    
    try:
        pdf_reader = pypdf.PdfReader(io.BytesIO(file_content))
        
        for page_num, page in enumerate(pdf_reader.pages, start=1):
            page_text = page.extract_text()
            if page_text:
                # Basic cleaning
                clean_text = re.sub(r'-\n', '', page_text)
                clean_text = re.sub(r'(?<!\n)\n(?!\n)', ' ', clean_text)
                clean_text = re.sub(r'\s+', ' ', clean_text).strip()
                
                if clean_text:
                    pages_data.append((file_path, page_num, clean_text))
                    
    except Exception as e:
        pass # Silently handle unreadable PDFs
        
    return pages_data

# 2. Function to split text into overlapping chunks
def create_chunks(page_data, chunk_size, chunk_overlap):
    """
    Splits page text into chunks of specified word count with overlap.
    Returns a list of dictionaries containing the chunk and its metadata.
    """
    file_path, page_num, text = page_data
    words = text.split()
    chunks = []
    
    step = chunk_size - chunk_overlap
    if step <= 0:
        step = chunk_size
        
    chunk_index = 1
    
    for i in range(0, len(words), step):
        chunk_words = words[i:i + chunk_size]
        
        if len(chunk_words) < 20 and i > 0:
            break
            
        chunk_text = " ".join(chunk_words)
        unique_chunk_id = f"{file_path.split('/')[-1]}_p{page_num}_c{chunk_index}"
        
        chunk_record = {
            "id": unique_chunk_id,
            "source_pdf": file_path.split('/')[-1],
            "page_num": page_num,
            "chunk_id": unique_chunk_id,
            "start_word": i,
            "end_word": i + len(chunk_words),
            "chunk_text": chunk_text
        }
        
        chunks.append(chunk_record)
        chunk_index += 1
        
    return chunks

# 3. Generator function for SBERT embeddings via mapPartitions
def generate_embeddings_sbert(partition):
    """
    Generator function to compute SBERT embeddings for a partition of chunks.
    """
    from sentence_transformers import SentenceTransformer
    
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    for chunk in partition:
        embedding_vector = model.encode(chunk['chunk_text']).tolist()
        
        # Add the embedding to the existing chunk dictionary
        chunk["embedding"] = embedding_vector
        chunk["embedding_strategy"] = "SBERT"
        
        yield chunk


In [33]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("BigData_RAG_Task1") \
    .master(f"local[{parallelism}]") \
    .getOrCreate()
sc = spark.sparkContext

print(f"Starting pipeline with {embedding_strategy} strategy...")

# Step 1: Load binary files
pdf_rdd = sc.binaryFiles(input_dir)

# Step 2: Extract pages
pages_rdd = pdf_rdd.flatMap(extract_pages_from_pdf)

# Step 3: Chunking
chunks_rdd = pages_rdd.flatMap(lambda page: create_chunks(page, chunk_size, chunk_overlap))

# Step 4: Embeddings
embedded_rdd = chunks_rdd.mapPartitions(generate_embeddings_sbert)

Starting pipeline with SBERT strategy...


In [34]:
# Convert to DataFrame
embeddings_df = spark.createDataFrame(embedded_rdd)

# Save to Parquet
embeddings_df.write.mode("overwrite").parquet(output_dir)
print(f"Successfully saved embedded data to {output_dir}")

# Verification: Show schema and count
embeddings_df.printSchema()
print(f"Total chunks processed: {embeddings_df.count()}")

/home/anischaibi/mon_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21512.46it/s]
/home/anischaibi/mon_env/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTor

Successfully saved embedded data to data/data_processed/embedded_chunks.parquet
root
 |-- chunk_id: string (nullable = true)
 |-- chunk_text: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- embedding_strategy: string (nullable = true)
 |-- end_word: long (nullable = true)
 |-- id: string (nullable = true)
 |-- page_num: long (nullable = true)
 |-- source_pdf: string (nullable = true)
 |-- start_word: long (nullable = true)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15664.57it/s]
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 72 0 (offset 0)
Ignoring wrong pointing object 7 0 (offset 0)                       (3 + 2) / 5]
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 11 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 27 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 37 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 61 0 (offset 0)
Ignoring wrong pointing object 71 0 (offset 0)
Ignoring wrong pointing object 73 0 (offset 0)
Ignoring wrong pointing object 79 0 (offset 0)
Ignoring wrong pointing object 96 0 (offset 0)
Igno

Total chunks processed: 2653
